# 01 - Exploratory Data Analysis (Insurance Pricing)

**Project:** End-to-End Insurance Pricing Engine on SageMaker  
**Dataset:** freMTPL2 French Motor Third-Party Liability

Goal: Understand the data, calculate claim frequency, analyze exposure, and identify key pricing variables.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
print("Current working directory:")
print(os.getcwd())


# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully")

In [ ]:
# Load the frequency dataset - Correct path from inside notebooks folder
df_freq = pd.read_csv('../data/raw/freMTPL2freq.csv')

print("✅ freMTPL2freq.csv loaded successfully!")
print(f"Shape: {df_freq.shape[0]:,} rows × {df_freq.shape[1]} columns")
print("\nFirst 5 rows:")
display(df_freq.head())

print("\nColumns:")
print(df_freq.columns.tolist())

# 01 - Exploratory Data Analysis

**Dataset**: freMTPL2 French Motor Third-Party Liability  
**Goal**: Understand the data structure and key risk drivers for insurance pricing.

In insurance pricing, understanding the distribution of the target (`Frequency`) and key risk factors is critical. 
Most policies have zero claims, which is typical for frequency modeling (zero-inflated data).

In [ ]:
# =============================================
# DISTRIBUTION ANALYSIS
# =============================================

print("=== DATASET SUMMARY ===\n")

total_policies = len(df_freq)
total_claims = df_freq['ClaimNb'].sum()
total_exposure = df_freq['Exposure'].sum()
overall_freq = total_claims / total_exposure

print(f"Total Policies          : {total_policies:,}")
print(f"Total Claims            : {total_claims:,}")
print(f"Total Exposure          : {total_exposure:,.2f} policy-years")
print(f"Overall Claim Frequency : {overall_freq:.4f}\n")

print(f"Mean Frequency          : {df_freq['Frequency'].mean():.4f}")
print(f"Median Frequency        : {df_freq['Frequency'].median():.4f}")
print(f"99th Percentile         : {df_freq['Frequency'].quantile(0.99):.4f}")
print(f"Max Frequency           : {df_freq['Frequency'].max():.4f}  ← Potential outlier")



In [ ]:


fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Claim Count Distribution (log scale for better visibility)
sns.countplot(data=df_freq, x='ClaimNb', ax=axes[0,0])
axes[0,0].set_title('Distribution of Claim Count (ClaimNb)')
axes[0,0].set_xlabel('Number of Claims')
axes[0,0].set_yscale('log')   # Log scale helps see small counts

# 2. Frequency Distribution (zoomed + log)
sns.histplot(data=df_freq, x='Frequency', bins=50, ax=axes[0,1])
axes[0,1].set_title('Claim Frequency Distribution')
axes[0,1].set_xlabel('Frequency')
axes[0,1].set_yscale('log')

# 3. Frequency < 2 (more useful view)
sns.histplot(data=df_freq[df_freq['Frequency'] < 2], x='Frequency', bins=30, ax=axes[0,2])
axes[0,2].set_title('Frequency Distribution (Frequency < 2)')
axes[0,2].set_xlabel('Frequency')

# 4. Exposure Distribution
sns.histplot(data=df_freq, x='Exposure', bins=50, ax=axes[1,0])
axes[1,0].set_title('Exposure Distribution')
axes[1,0].set_xlabel('Exposure (years)')

# 5. BonusMalus vs Frequency (better view - mean instead of boxplot with outliers)
bonus_freq = df_freq.groupby('BonusMalus')['Frequency'].mean().reset_index()
sns.barplot(data=bonus_freq, x='BonusMalus', y='Frequency', ax=axes[1,1])
axes[1,1].set_title('Average Frequency by BonusMalus')
axes[1,1].tick_params(axis='x', rotation=45)

# 6. Driver Age vs Frequency
sns.boxplot(data=df_freq, x=pd.cut(df_freq['DrivAge'], bins=8), y='Frequency', ax=axes[1,2])
axes[1,2].set_title('Frequency by Driver Age Group')
axes[1,2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 2. Univariate Risk Factor Analysis

Below we analyze how claim frequency varies across key rating factors. 
These patterns are critical for GLM modeling and help us understand which variables have strong predictive power.

In [ ]:
# Univariate analysis of key pricing variables
key_variables = ['BonusMalus', 'DrivAge', 'VehAge', 'VehPower', 'Area', 'VehGas', 'Density']

print("=== AVERAGE CLAIM FREQUENCY BY KEY VARIABLES ===\n")

for var in key_variables:
    if var in df_freq.columns:
        # Group by variable and calculate mean frequency + exposure-weighted frequency
        grouped = df_freq.groupby(var).agg(
            Avg_Frequency=('Frequency', 'mean'),
            Total_Exposure=('Exposure', 'sum'),
            Total_Claims=('ClaimNb', 'sum'),
            Policy_Count=('Frequency', 'count')
        ).round(4)
        
        print(f"\n{var} (showing top 10 rows):")
        print(grouped.head(10))
        print("-" * 60)

### Key Observations 
- Total policies: ~678k
- Overall claim frequency: ~0.264 claims per policy-year
- Most policies have zero claims (median frequency = 0)
- Clear risk patterns in `BonusMalus`, `DrivAge`, `VehAge`, and `Area`

### Key Observations from Univariate Analysis

As a Data Scientist with 7+ years of experience building GLM pricing models for insurance, here are the main insights from the risk factor analysis:

#### 1. BonusMalus (Strong Predictive Power)
- Clear increasing trend in frequency as BonusMalus increases.
- Some levels (52, 56, 58) show notable spikes — this is expected behavior since BonusMalus reflects past claims history.
- This variable should be treated carefully in modeling (likely as categorical or with splines).

#### 2. Driver Age (Classic Risk Pattern)
- Young drivers (ages 18–23) have significantly higher claim frequency (0.44 – 0.70).
- Frequency generally decreases with age, which aligns with traditional actuarial understanding of driver risk.
- This is a core rating factor in almost all motor pricing models.

#### 3. Vehicle Age (Interesting Pattern)
- Brand new vehicles (`VehAge = 0`) show unusually high frequency (1.14).
- This could be due to:
  - Higher mileage in the first year
  - More aggressive driving with new cars
  - Potential data reporting bias
- Worth investigating further during feature engineering.

#### 4. Area & Density (Urban/Rural Effect)
- Frequency increases noticeably from rural Area `A` (0.218) to urban Area `F` (0.406).
- This confirms the expected geographic risk gradient.

#### 5. Vehicle Gas Type
- Regular gasoline vehicles have substantially higher frequency (0.316) compared to Diesel (0.210).
- This may be correlated with vehicle type, power, or driving behavior.

#### 6. Overall Data Characteristics
- The dataset is heavily zero-inflated (median frequency = 0.0).
- Maximum frequency of 732 is a clear outlier that should be investigated or capped during data cleaning.
- Total exposure of ~358,499 policy-years provides a solid volume for reliable modeling.

These patterns will heavily influence our choice of modeling approach (GLM baseline vs XGBoost vs Neural Nets).